<a href="https://colab.research.google.com/github/GustavoFTeo/INFO3-BD2/blob/main/SQLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Forma mais direta
import sqlite3
connection = sqlite3.connect("pizza_app.sqlite")

In [2]:
from sqlite3 import Error

def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)

        # commit necessário para alterações no banco
        connection.commit() ###

        print(f"Query executada.")
        if cursor.rowcount != -1:
            print(f"{cursor.rowcount} linha(s) afetadas")

    except Error as e:
        print(f"Erro: '{e}'")

In [3]:
# Cria a tabela produto #
create_produto_table = \
"""CREATE TABLE produto (
    id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo VARCHAR(50),
    desc_item VARCHAR(100),
    vl_preco DECIMAL(10, 2)
);"""

execute_query(connection, create_produto_table)
#########################

# Cria a tabela pedido #
create_pedido_table = \
"""CREATE TABLE pedido (
    id_pedido INTEGER PRIMARY KEY AUTOINCREMENT,
    dt_pedido DATE,
    fl_ketchup BOOLEAN,
    desc_uf CHAR(2),
    txt_recado TEXT
);"""

execute_query(connection, create_pedido_table)
#########################

# Cria a tabela item_pedido #
create_item_pedido_table = \
"""CREATE TABLE item_pedido (
    id_pedido INT NOT NULL,
    id_produto INT NOT NULL,
    quantidade INT NOT NULL,
    PRIMARY KEY (id_pedido, id_produto),
    FOREIGN KEY (id_pedido) REFERENCES pedido(id_pedido),
    FOREIGN KEY (id_produto) REFERENCES produto(id_produto)
);"""
execute_query(connection, create_item_pedido_table)
#########################

Query executada.
Query executada.
Query executada.


In [4]:
## Inserindo registros manualmente

# Inserindo produto #
insert_produto = \
"""INSERT INTO
produto (tipo, desc_item, vl_preco)
VALUES
('ingrediente', 'camarão', 6),
('massa', 'tradicional', 9.25),
('borda', 'tradicional', 0),
('queijo', 'muçarela', 4),
('bebida', 'refrigerante', 5);
"""
execute_query(connection, insert_produto)
######################

# Inserindo pedido
insert_pedido = \
"""INSERT INTO
pedido (dt_pedido, fl_ketchup, desc_uf, txt_recado)
VALUES
('2023-06-01', TRUE, 'MG', 'Capricha no queijo!');
"""
execute_query(connection, insert_pedido)
######################

Query executada.
5 linha(s) afetadas
Query executada.
1 linha(s) afetadas


In [5]:
# Inserindo item_pedido
itens = (
    {'id_pedido': 1, 'id_produto': 2, 'qtd': 1},
    {'id_pedido': 1, 'id_produto': 3, 'qtd': 1},
    {'id_pedido': 1, 'id_produto': 1, 'qtd': 1},
    {'id_pedido': 1, 'id_produto': 4, 'qtd': 2},
    {'id_pedido': 1, 'id_produto': 5, 'qtd': 3}
)

insert_item_pedido = \
"""INSERT INTO item_pedido (id_pedido, id_produto, quantidade)
VALUES(:id_pedido, :id_produto, :qtd);"""

cursor = connection.cursor()
cursor.executemany(insert_item_pedido, itens)
connection.commit() # necessário para inserções
cursor.close()

In [6]:
def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()

        return result
    except Error as e:
        print(f"Erro: '{e}'")

In [7]:
tabela = 'produto'
query = f"SELECT * FROM {tabela}"
resultado = execute_read_query(connection, query)

print(f"Tabela: {tabela}")
for res in resultado:
    print(res)

Tabela: produto
(1, 'ingrediente', 'camarão', 6)
(2, 'massa', 'tradicional', 9.25)
(3, 'borda', 'tradicional', 0)
(4, 'queijo', 'muçarela', 4)
(5, 'bebida', 'refrigerante', 5)


In [8]:
select_table_names = \
"SELECT name FROM sqlite_schema WHERE type='table';"
tables = execute_read_query(connection, select_table_names)
print(tables, '\n')

for table in tables:
    select_all = f"SELECT * FROM {table[0]}"
    res = execute_read_query(connection, select_all)
    print(f"{table[0]}: {res}")

[('sqlite_sequence',), ('produto',), ('pedido',), ('item_pedido',)] 

sqlite_sequence: [('produto', 5), ('pedido', 1)]
produto: [(1, 'ingrediente', 'camarão', 6), (2, 'massa', 'tradicional', 9.25), (3, 'borda', 'tradicional', 0), (4, 'queijo', 'muçarela', 4), (5, 'bebida', 'refrigerante', 5)]
pedido: [(1, '2023-06-01', 1, 'MG', 'Capricha no queijo!')]
item_pedido: [(1, 2, 1), (1, 3, 1), (1, 1, 1), (1, 4, 2), (1, 5, 3)]


In [9]:
execute_query(connection, "DELETE FROM item_pedido;")
execute_query(connection, "DELETE FROM pedido;")
execute_query(connection, "DELETE FROM produto;")

Query executada.
5 linha(s) afetadas
Query executada.
1 linha(s) afetadas
Query executada.
5 linha(s) afetadas


In [10]:
! wget https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/pizza_query/item_pedido.csv
! wget https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/pizza_query/pedido.csv
! wget https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/pizza_query/produto.csv

--2026-06-26 19:49:54--  https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/pizza_query/item_pedido.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 103557 (101K) [text/plain]
Saving to: ‘item_pedido.csv.1’

item_pedido.csv.1   100%[===================>] 101.13K  --.-KB/s    in 0.02s   

2026-06-26 19:49:55 (4.91 MB/s) - ‘item_pedido.csv.1’ saved [103557/103557]

--2026-06-26 19:49:55--  https://raw.githubusercontent.com/camilalaranjeira/python-intermediario/main/pizza_query/pedido.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awai

In [11]:
import pandas as pd
df_pedido = pd.read_csv(f'pedido.csv')
display(df_pedido.head())

,id_pedido,dt_pedido,fl_ketchup,desc_uf,txt_recado
0,0,2023-05-11,NaN,GO,NaN
1,1,2023-05-11,NaN,PR,Aquela pizza perfeita! :-D
2,2,2023-05-11,NaN,SP,Muito obrigado!!
3,3,2023-05-11,NaN,SP,NaN
4,4,2023-05-11,NaN,RS,Capricha no peperoni


In [12]:
cursor = connection.cursor()
cursor.execute(f"SELECT * FROM pedido WHERE id_pedido < 5;")
result = cursor.fetchall()

In [13]:
df_pedido.to_sql('pedido', connection, if_exists='append', index=False)

count_rows = "SELECT COUNT(id_pedido) as count_id FROM pedido;"
print(execute_read_query(connection, count_rows))

select_all = f"SELECT * FROM pedido WHERE id_pedido < 5;"
execute_read_query(connection, select_all)

[(1106,)]


[(0, '2023-05-11', None, 'GO', None),
 (1, '2023-05-11', None, 'PR', 'Aquela pizza perfeita! :-D'),
 (2, '2023-05-11', None, 'SP', 'Muito obrigado!!'),
 (3, '2023-05-11', None, 'SP', None),
 (4, '2023-05-11', None, 'RS', 'Capricha no peperoni')]

In [14]:
# df_pedido.to_sql('pedido', connection, if_exists='replace', index=False)
res = execute_read_query(connection, "SELECT sql FROM sqlite_schema")
for r in res:
    print(r[0])

CREATE TABLE sqlite_sequence(name,seq)
CREATE TABLE produto (
    id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo VARCHAR(50),
    desc_item VARCHAR(100),
    vl_preco DECIMAL(10, 2)
)
CREATE TABLE pedido (
    id_pedido INTEGER PRIMARY KEY AUTOINCREMENT,
    dt_pedido DATE,
    fl_ketchup BOOLEAN,
    desc_uf CHAR(2),
    txt_recado TEXT
)
CREATE TABLE item_pedido (
    id_pedido INT NOT NULL,
    id_produto INT NOT NULL,
    quantidade INT NOT NULL,
    PRIMARY KEY (id_pedido, id_produto),
    FOREIGN KEY (id_pedido) REFERENCES pedido(id_pedido),
    FOREIGN KEY (id_produto) REFERENCES produto(id_produto)
)
None


In [15]:
!pip install sqlmodel

In [16]:
from datetime import datetime
from sqlmodel import (
    SQLModel, Field, Relationship,
    create_engine, Session, select
)

## Definição de classes/tabelas
class Equipe(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    nome: str
    pessoas: list["Pessoa"] = Relationship(back_populates="equipe")

class Pessoa(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    nome: str
    apelido: str
    data_nasc: datetime

    id_equipe: int | None = Field(default=None, foreign_key="equipe.id")
    equipe: Equipe | None = Relationship(back_populates="pessoas")

# Iniciando a conexão com o banco (SQLite)
sqlite_file_name = "database.db"
sqlite_url = f"sqlite:///{sqlite_file_name}"

engine = create_engine(sqlite_url, echo=True)

# Criando as tabelas
SQLModel.metadata.create_all(engine)

# Inserindo dados (isso estaria em uma função)
with Session(engine) as session:
    pessoa1 = Pessoa(nome="André Augusto", apelido="Melhor amigo",
                      data_nasc=datetime(2007, 2, 8))
    pessoa2 = Pessoa(nome="Gabriel Felipe", apelido="Tilipe",
                      data_nasc=datetime(2005, 10, 19))

    session.add(pessoa1)
    session.add(pessoa2)
    session.commit()

# Fazendo consultas
with Session(engine) as session:
    statement = select(Pessoa).where(Pessoa.apelido == "Tilipe")
    pessoa = session.exec(statement).first()
    print(pessoa.id, pessoa.nome)

2026-06-26 19:50:08,904 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-06-26 19:50:08,910 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("equipe")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("equipe")


2026-06-26 19:50:08,911 INFO sqlalchemy.engine.Engine [raw sql] ()


...

INFO:sqlalchemy.engine.Engine:SELECT pessoa.id, pessoa.nome, pessoa.apelido, pessoa.data_nasc, pessoa.id_equipe 
FROM pessoa 
WHERE pessoa.apelido = ?


2026-06-26 19:50:08,965 INFO sqlalchemy.engine.Engine [generated in 0.00154s] ('Tilipe',)


INFO:sqlalchemy.engine.Engine:[generated in 0.00154s] ('Tilipe',)


2 Gabriel Felipe
2026-06-26 19:50:08,969 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


In [17]:
df_pedido.to_sql('pedido', connection, if_exists='replace', index=False)
res = execute_read_query(connection, "SELECT sql FROM sqlite_schema")
for r in res:
    print(r[0])

CREATE TABLE sqlite_sequence(name,seq)
CREATE TABLE produto (
    id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo VARCHAR(50),
    desc_item VARCHAR(100),
    vl_preco DECIMAL(10, 2)
)
CREATE TABLE item_pedido (
    id_pedido INT NOT NULL,
    id_produto INT NOT NULL,
    quantidade INT NOT NULL,
    PRIMARY KEY (id_pedido, id_produto),
    FOREIGN KEY (id_pedido) REFERENCES pedido(id_pedido),
    FOREIGN KEY (id_produto) REFERENCES produto(id_produto)
)
None
CREATE TABLE "pedido" (
"id_pedido" INTEGER,
  "dt_pedido" TEXT,
  "fl_ketchup" INTEGER,
  "desc_uf" TEXT,
  "txt_recado" TEXT
)


In [18]:
query="""
SELECT desc_uf, COUNT(*) as count_pedidos
FROM pedido
GROUP BY desc_uf
ORDER BY count_pedidos DESC
LIMIT 5
"""
pd.read_sql_query(query, connection)

,desc_uf,count_pedidos
0,SP,395
1,RJ,103
2,MG,92
3,PR,76
4,RS,50
